In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["Deepseek_API_KEY"]=os.getenv("Deepseek_API_KEY")
print("Keys loaded!")


Keys loaded!


STATE Backend


In [2]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend


In [6]:
agent = create_deep_agent(model="deepseek-v4-pro")

#under the hood this is what the agent doing- explicit statebackend

agent2 = create_deep_agent(
    model="deepseek-v4-pro",
    backend=StateBackend(),
)




In [9]:
#here i invoke the agent to write the file a file 
#and statebackend keeps that file inside-langraph state, after that 
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply --------------------")
print(result["messages"][-1].content)


--- Agent reply --------------------
I've created the file at `/notes/todo.txt` with the requested content.


how to check the file,,, it will store at ram, 

In [10]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='bd39266a-046b-494b-a0ff-06002f40fc88'),
  AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': 'The user wants me to create a file at /notes/todo.txt with specific content. Let me just write the file.'}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 2835, 'total_tokens': 2939, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 25, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 2816, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 2816, 'prompt_cache_miss_tokens': 19}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-pro

In [ ]:
# ---------------------------------------------------------------------------
# 3. CHECK the backend is working
#    With StateBackend, written files appear under result["files"]
# ---------------------------------------------------------------------------
print("\n--- Backend check ------------------------------------------")
files = result.get("files", {})

if files:
    print(f" StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")


--- Backend check ------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video\n', 'encoding': 'utf-8', 'created_at': '2026-09-28T19:49:41.359834+00:00', 'modified_at': '2026-09-28T19:49:41.359834+00:00'}


In [12]:
# ---------------------------------------------------------------------------
# 4. Prove persistence WITHIN the same thread:
#    feed the returned state back in and ask it to READ the file
# ---------------------------------------------------------------------------
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),  # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) --------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) --------------------------
Here's the content of `/notes/todo.txt`:

```
1. Record video
2. Edit video
3. Upload video
```


###FilesystemBackend(local disk)

In [ ]:
# ---------------------------------------------------------------------------
# 1. Create the agent with a real-disk backend
#    root_dir="." -> files land relative to your current working directory
#    virtual_mode=True -> agent uses virtual paths like /notes/todo.txt,
#                         mapped onto root_dir
# ---------------------------------------------------------------------------
from deepagents.backends import FilesystemBackend
ROOT= "."
agent= create_deep_agent(model="deepseek-v4-pro", backend=FilesystemBackend(root_dir=ROOT, virtual_mode= True))

print(f"agent created with filesystembackend(root_dir={ROOT!r})")
print(" files written by the agent will appear on your actual disk")

agent created with filesystembackend(root_dir='.')
 files written by the agent will appear on your actual disk


In [20]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply --------------------")
print(result["messages"][-1].content)


--- Agent reply --------------------
Done! I've created the file at `/notes/todo.txt` with exactly the content you requested.


In [21]:
# 5. Prove persistence ACROSS sessions:
# with filesystembackend, this file survives even after Python exits.
# A brand-new agent (fresh state) can read it back from disk.
# ---------------------------------------------------------------------------
fresh_agent = create_deep_agent(
    model="deepseek-v4-pro",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in - the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) -------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) -------
1. Record video
2. Edit video
3. Upload video


"""
Deep Agent - StoreBackend verification
--------------------------------------
Creates a deep agent backed by a LangGraph store, invokes it to write a 
file on one thread, then proves the backend works by reading that file 
back on a DIFFERENT thread - something StateBackend cannot do.
"""

In [23]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="deepseek-v4-pro",
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.


In [26]:
import uuid

# 2. THREAD 1 - write a file
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Make sure to use the agent that has the FilesystemBackend!
# If your disk-based agent is named 'fs_agent' or 'fresh_agent', change 'agent' below to that name.
result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) -----------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) -----------------------------
Done! I've created the file at `/notes/todo.txt` with exactly the content you specified.


In [27]:
import uuid

# 3. THREAD 2 - read the file in a brand new thread
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result_2 = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Read /notes/todo.txt back to me."
        }]
    },
    config=thread_2,
)

print("\n--- Agent reply (thread 2) -----------------------------")
print(result_2["messages"][-1].content)


--- Agent reply (thread 2) -----------------------------
Here's what's in `/notes/todo.txt`:

1. Record video
2. Edit video
3. Upload video


With StoreBackend + InMemoryStore, the file is not saved to disk at all. It lives in RAM, inside the InMemoryStore object, as an entry keyed under the namespace.

## Deep Agent Backends — Where Your Files Actually Live

Every Deep Agent works with a **virtual filesystem**: its tools read and write files using paths like `/notes/todo.txt`. But those paths are an abstraction. The **backend** decides where that data physically lives — and that single choice changes everything about persistence, sharing, and durability.

The agent code stays identical across all three. Only the backend changes.

---

### 1. StateBackend (default)

Files live **in the agent's LangGraph state** — i.e. in RAM, tied to a single thread. They're available during the run via `result["files"]`, and you must manually carry that state forward to keep using them.

- **Where:** in-memory dict, inside one thread's state
- **Survives:** within a single conversation/thread only
- **Gone when:** the thread ends or the process exits
- **Use it for:** ephemeral scratch space, temporary working files

### 2. FilesystemBackend

Files are written **directly to your actual hard drive**. You define a `root_dir` (like `.` or `D:\deep agent`), and the virtual paths like `/notes/todo.txt` are mapped onto real folders.

- **Where:** on your real disk (e.g., `root_dir/notes/todo.txt`)
- **Survives:** forever, even if you restart the kernel or close VS Code
- **Gone when:** you manually delete the file
- **Use it for:** real research outputs, saving data, sharing files between threads

### 3. StoreBackend + InMemoryStore

Files live in **RAM, inside an InMemoryStore object**. This is useful when you want a persistent scratch space that can be shared across multiple threads in the same session, without cluttering your hard drive.

- **Where:** inside an `InMemoryStore` object in RAM
- **Survives:** across different threads, as long as the kernel is alive
- **Gone when:** you restart the Python kernel
- **Use it for:** cross-thread working memory without disk writes